# Chatbot con MCP, Ollama y Gradio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/3-chatbot-con-mcp-y-gradio.ipynb)

Este notebook es un ejemplo avanzado y opcional de la sesión. No introduce conceptos nuevos: simplemente toma el agente con herramientas MCP del notebook anterior y le monta encima una interfaz conversacional con Gradio, siguiendo el estilo visto antes en la sesión de RAG con LangChain. La idea es cerrar la unidad mostrando cómo pasar de una integración técnica a una experiencia de usuario lista para demostración.

### Referencias
- [Gradio](https://www.gradio.app/)
- [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://pypi.org/project/langchain-mcp-adapters/)
- [Ollama](https://ollama.com/)


In [4]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
!test '{IN_COLAB}' = 'True' && pip install "mcp>=1.24.0,<2.0.0" "langchain-mcp-adapters>=0.3.0,<0.4.0" langchain langchain-core langchain-ollama langgraph httpx gradio ollama colab-xterm

# En local, instala/actualiza con: pip install -r requirements.txt

### Cargando a Ollama

Usaremos el mismo modelo local y la misma idea de tools de los notebooks previos para que lo único nuevo aquí sea la interfaz.


In [ ]:
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


## Atención

En Colab recuerda arrancar `ollama serve` desde la terminal embebida si todavía no está corriendo. En local basta con tener el servicio levantado de antemano.

In [ ]:
%load_ext colabxterm
%xterm


Mantendremos `llama3.2:3b` para la demostración.


In [ ]:
!ollama pull llama3.2:3b


## Reutilizamos los mismos servidores MCP

Para que el notebook sea transparente y fácil de depurar, **no** ocultamos la implementación en cadenas largas. Reutilizamos los mismos archivos Python del repositorio para calculadora y clima, igual que en el notebook anterior.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / 'Sesion6' / 'mcp_servers').exists():
    SERVERS_DIR = REPO_ROOT / 'Sesion6' / 'mcp_servers'
else:
    SERVERS_DIR = REPO_ROOT / 'mcp_servers'

calculator_server = SERVERS_DIR / 'calculator_mcp_server.py'
weather_server = SERVERS_DIR / 'weather_mcp_server.py'

assert calculator_server.exists(), f'No existe: {calculator_server}'
assert weather_server.exists(), f'No existe: {weather_server}'

SERVERS_DIR

In [ ]:
print(f'Servidor calculadora: {calculator_server}')
print('-' * 80)
print(calculator_server.read_text(encoding='utf-8'))

In [ ]:
print(f'Servidor clima: {weather_server}')
print('-' * 80)
print(weather_server.read_text(encoding='utf-8'))

## Inicio y apagado de servidores MCP

Tienes dos formas válidas de ejecutar los servidores:

1. Modo automático (recomendado aquí): el cliente MCP por `stdio` levanta y cierra procesos durante la llamada.
2. Modo manual: inicia cada servidor en terminal separada y detén con `Ctrl+C` al finalizar.

Si usas modo manual, recuerda apagar procesos para evitar sesiones colgadas.

In [ ]:
import sys

manual_start_commands = [
    f"{sys.executable} {calculator_server}",
    f"{sys.executable} {weather_server}",
]

print('Comandos para modo manual (ejecutar en terminales separadas):')
for cmd in manual_start_commands:
    print(f'- {cmd}')

print('\nPara detener cada servidor manual: Ctrl+C en su terminal.')

## Construimos el agente consumidor de herramientas MCP

Igual que antes, el agente no conoce directamente las funciones de Python; descubre las capacidades a través de los servidores MCP y decide cuándo invocarlas.

### Cómo decide el agente usar tools

En modo autónomo, el modelo puede responder directamente o llamar tools. Que una tool esté disponible **no garantiza** su uso en todos los turnos.

Para mejorar la probabilidad de uso de tool sin forzar:

1. Escribe prompts con intención operativa clara (por ejemplo: "usa la herramienta de clima").
2. Define descripciones de tool precisas y accionables.
3. Usa instrucciones de sistema que prioricen tools cuando haya datos externos.
4. Mantén una ruta guiada/determinística para tareas donde sí necesitas evidencia MCP obligatoria.

In [9]:
import re
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

client = MultiServerMCPClient(
    {
        'calculadora': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(calculator_server)],
        },
        'clima': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(weather_server)],
        },
    }
)

mcp_tools = await client.get_tools()
agent_mcp = create_react_agent(model=llm, tools=mcp_tools)

def available_tool_names():
    return [tool.name for tool in mcp_tools]

def tool_from_name(tool_name: str):
    return next((tool for tool in mcp_tools if tool.name == tool_name), None)

def detect_suggested_tool(question: str) -> str | None:
    lower = question.lower()
    weather_signals = ['clima', 'temperatura', 'humedad', 'viento', 'weather', 'lluvia']
    if any(word in lower for word in weather_signals):
        return 'clima_actual'
    has_digit = any(ch.isdigit() for ch in question)
    has_operator = any(op in question for op in ['+', '-', '*', '/', '%', '(', ')'])
    if has_digit and has_operator:
        return 'calculadora'
    return None

def select_tool_name(question: str, preferred_tool: str):
    if preferred_tool and preferred_tool != 'Auto':
        return preferred_tool
    return detect_suggested_tool(question)

def infer_calculator_expression(question: str) -> str | None:
    # Extrae una expresión aritmética probable desde el texto del usuario.
    candidates = re.findall(r"[0-9\s\+\-\*\/%\(\)\.]+", question)
    candidates = [c.strip() for c in candidates if any(ch.isdigit() for ch in c) and any(op in c for op in '+-*/%')]
    if not candidates:
        return None
    return max(candidates, key=len)

def infer_city(question: str) -> str:
    # Heurística simple: toma texto tras 'en ...' y limpia signos frecuentes.
    match = re.search(r"\ben\s+([^\?\.!\n]+)", question, flags=re.IGNORECASE)
    if match:
        city = match.group(1).strip(' ,;')
        if city:
            return city
    return 'Cali, Colombia'

async def run_selected_tool(tool_name: str, question: str):
    tool = tool_from_name(tool_name)
    if tool is None:
        raise ValueError(f'Herramienta no encontrada: {tool_name}')

    if tool_name == 'calculadora':
        expression = infer_calculator_expression(question)
        if not expression:
            raise ValueError(
                'No pude inferir una expresión aritmética del mensaje. '
                'Escribe algo como: "¿Cuánto es (125 * 17) + 938?".'
            )
        payload = {'expression': expression}
    elif tool_name == 'clima_actual':
        payload = {'city': infer_city(question)}
    else:
        raise ValueError(f'Herramienta no soportada: {tool_name}')

    return await tool.ainvoke(payload)

def tool_message_used(messages) -> bool:
    return any(getattr(msg, 'type', '') == 'tool' for msg in messages)

async def synthesize_from_tool(question: str, tool_name: str, tool_output):
    synthesis = llm.invoke([
        ('system', 'Responde en español, de forma breve y usando solamente el resultado de la herramienta.'),
        ('user', f"Pregunta: {question}\n\nSalida de {tool_name}: {tool_output}\n\nRedacta una respuesta final clara."),
    ])
    return synthesis.content

## Del agente a la interfaz conversacional

Aquí reaparece una idea de la sesión de RAG: Gradio maneja historial, mientras reconstruimos mensajes para el agente.

En esta versión añadimos controles de estrategia para clase:

- Autónomo: el agente decide libremente.
- Guiado: sugerimos tool preferida y permitimos fallback opcional.
- Determinístico: ejecutamos una tool explícita y luego redactamos la respuesta final.

Nota de diseño de UX: el único input principal es el chat. No pedimos campos separados para cálculo o clima porque la intención idealmente se infiere desde el mensaje del usuario. El selector de tool funciona como preferencia/guía, no como un formulario obligatorio.

In [10]:
import gradio as gr

def history_to_messages(question, chat_history):
    messages = []
    for item in (chat_history or []):
        role = item.get('role')
        content = item.get('content')
        if role in {'user', 'assistant'} and content is not None:
            messages.append((role, str(content)))
    messages.append(('user', question))
    return messages

def normalize_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get('text') or item.get('content') or str(item))
            else:
                parts.append(str(item))
        return '\n'.join(parts)
    return str(content)

def compact_trace_text(trace: dict):
    lines = [
        f"modo={trace.get('mode')}",
        f"used_tool={trace.get('used_tool')}",
        f"tool_selected={trace.get('tool_selected')}",
        f"guided_fallback_used={trace.get('guided_fallback_used')}",
    ]
    if trace.get('tool_error'):
        lines.append(f"tool_error={trace.get('tool_error')}")
    return ' | '.join(lines)

async def respond(
    question,
    chat_history,
    mode,
    preferred_tool,
    guided_fallback,
    show_trace,
 ):
    chat_history = chat_history or []
    mode = mode or 'Guided'
    messages = history_to_messages(question, chat_history)
    selected_tool = select_tool_name(question, preferred_tool)
    trace = {
        'mode': mode,
        'tool_selected': selected_tool,
        'used_tool': False,
        'guided_fallback_used': False,
        'tool_error': None,
    }

    try:
        if mode == 'Deterministic':
            if selected_tool is None:
                answer = (
                    'No pude inferir una herramienta automáticamente. '
                    'Selecciona una herramienta o ajusta tu pregunta.'
                )
            else:
                tool_output = await run_selected_tool(selected_tool, question)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True

        elif mode == 'Guided':
            guided_messages = messages
            if selected_tool is not None:
                guided_messages = [
                    (
                        'system',
                        f"Si la pregunta requiere cálculo o datos externos, prioriza usar la herramienta {selected_tool} antes de responder.",
                    )
                ] + guided_messages

            result = await agent_mcp.ainvoke({'messages': guided_messages})
            used_tool = tool_message_used(result['messages'])
            answer = normalize_content(result['messages'][-1].content)
            trace['used_tool'] = used_tool

            if guided_fallback and (not used_tool) and selected_tool is not None:
                tool_output = await run_selected_tool(selected_tool, question)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True
                trace['guided_fallback_used'] = True

        else:
            result = await agent_mcp.ainvoke({'messages': messages})
            trace['used_tool'] = tool_message_used(result['messages'])
            answer = normalize_content(result['messages'][-1].content)

    except Exception as exc:
        trace['tool_error'] = str(exc)
        answer = f"Ocurrió un error: {exc}"

    if show_trace:
        answer = f"{answer}\n\n---\nTraza MCP: {compact_trace_text(trace)}"

    updated_history = chat_history + [
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ]
    return '', updated_history

def reset_chat():
    return '', []

## Lanzando la interfaz de chat

Esta versión es opcional precisamente porque ya no enseña un concepto nuevo de NLP o de MCP; enseña cómo empaquetar el agente en una demo usable. Es útil para estudiantes que quieran presentar el flujo completo de extremo a extremo.

In [12]:
with gr.Blocks() as gr_blocks:
    gr.Markdown('## Chat con herramientas MCP')

    mode = gr.Dropdown(
        choices=['Guided', 'Autonomous', 'Deterministic'],
        value='Guided',
        label='Modo de ejecución',
    )
    preferred_tool = gr.Dropdown(
        choices=['Auto', 'calculadora', 'clima_actual'],
        value='Auto',
        label='Tool preferida',
    )
    guided_fallback = gr.Checkbox(
        label='En modo Guided, usar fallback de tool si el agente no la invoca',
        value=True,
    )
    show_trace = gr.Checkbox(
        label='Mostrar traza de uso MCP',
        value=False,
    )

    chatbot = gr.Chatbot(label='Historial')
    msg = gr.Textbox(
        label='¿Qué quieres preguntar?',
        placeholder='Ejemplo: ¿Cuál es la temperatura actual en Cali? o ¿Cuánto es (125 * 17) + 938?',
    )
    clear = gr.Button('Limpiar')

    msg.submit(
        respond,
        [
            msg,
            chatbot,
            mode,
            preferred_tool,
            guided_fallback,
            show_trace,
        ],
        [msg, chatbot],
    )
    clear.click(reset_chat, None, [msg, chatbot], queue=False)

gr_blocks.launch(inline=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
gr_blocks.close()


## Conclusiones

- Este notebook solo agrega una capa de presentación; el corazón sigue siendo el mismo agente con herramientas MCP.
- Gradio permite convertir el experimento técnico en una demo conversacional muy rápidamente.
- Como material opcional, ayuda a cerrar la sesión con una visión más aplicada sin sobrecargar el flujo conceptual principal.
